importing necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import xgboost as xgb

from google.colab import files
uploaded = files.upload()

: 

Data Loading

In [ ]:
train = pd.read_csv('Hackathon_Train.csv')
valid = pd.read_csv('Hackathon_Validation.csv')

print("Data loaded successfully")
print("Train shape:", train.shape)
print("Validation shape:", valid.shape)

Clean Engagement Columns

In [ ]:
engagement_cols = ['Views', 'Likes', 'Shares', 'Comments']

for col in engagement_cols:
    train[col] = train[col].astype(str).str.extract(r'([-+]?\d+)')[0]
    train[col] = pd.to_numeric(train[col], errors='coerce')

    train[col] = train[col].abs()

    train[col] = train[col].fillna(0)

    viral_median = train[(train['Viral']==1) & (train[col]>0)][col].median()
    nonviral_median = train[(train['Viral']==0) & (train[col]>0)][col].median()

    train.loc[(train['Viral']==1) & (train[col]==0), col] = viral_median
    train.loc[(train['Viral']==0) & (train[col]==0), col] = nonviral_median

print("Engagement columns cleaned and processed")
print(train[engagement_cols].head(10))


Descriptive Statistics for Engagement Metrics

In [ ]:

engagement_cols = ['Views', 'Likes', 'Shares', 'Comments']

print("Descriptive Statistics for Engagement Metrics:")
display(train[engagement_cols].describe().T)


Code: Summarize & Visualize Engagement

In [ ]:
import matplotlib.pyplot as plt

engagement_cols = ['Views', 'Likes']
main_platforms = ['youtube', 'tiktok', 'instagram', 'twitter']

for metric in engagement_cols:
    plt.figure(figsize=(8,5))

    train_main = train[train['Platform'].isin(main_platforms)]

    grouped = train_main.groupby(['Platform','Viral'])[metric].sum().reset_index()

    pivot_data = grouped.pivot(index='Platform', columns='Viral', values=metric)

    pivot_data.plot(kind='bar', figsize=(8,5), color=['blue','red'])
    plt.yscale('log')
    plt.xlabel('Platform')
    plt.ylabel(f'Total {metric} (log scale)')
    plt.title(f'Total {metric} by Platform: Viral vs Non-Viral')
    plt.legend(['Non-Viral','Viral'])
    plt.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
train_original = pd.read_csv('Hackathon_Train.csv')
train['Post_Date'] = train_original['Post_Date']

train['Post_Date'] = pd.to_datetime(train['Post_Date'], errors='coerce')

median_date = train['Post_Date'].median()
train['Post_Date'] = train['Post_Date'].fillna(median_date)

print("Post_Date cleaned and fixed")
print(train['Post_Date'].head(10))


# Part 2 – Engagement Metrics Cleaning

In [ ]:
categorical_cols = ['Platform', 'Content_Type', 'Region']

for col in categorical_cols:
    train[col] = train[col].astype(str).str.lower()

print("Categorical columns converted to lowercase")
print(train[categorical_cols].head(10))

Extract Date Features

In [ ]:
train['DayOfWeek'] = train['Post_Date'].dt.dayofweek
train['Month'] = train['Post_Date'].dt.month
train['Year'] = train['Post_Date'].dt.year
train['IsWeekend'] = train['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

print("Date features extracted")
print(train[['Post_Date','DayOfWeek','Month','Year','IsWeekend']].head(10))


Encode Categorical Columns

In [ ]:
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    train[col+'_encoded'] = le.fit_transform(train[col])
    le_dict[col] = le

print("Categorical columns encoded")
print(train[[c+'_encoded' for c in categorical_cols]].head(10))


 Engagement Ratios

In [ ]:
train['like_rate'] = train['Likes'] / train['Views']
train['share_rate'] = train['Shares'] / train['Views']
train['comment_rate'] = train['Comments'] / train['Views']

train[['like_rate','share_rate','comment_rate']] = train[['like_rate','share_rate','comment_rate']].fillna(0)

print("Engagement ratios calculated")
print(train[['like_rate','share_rate','comment_rate']].head(10))


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train[['like_rate','share_rate','comment_rate']] = scaler.fit_transform(
    train[['like_rate','share_rate','comment_rate']]
)


 Text Embeddings


In [ ]:
model_emb = SentenceTransformer("all-MiniLM-L6-v2")
text_list = train["Pseudo_Caption"].fillna("").astype(str).tolist()

print("Generating sentence embeddings...")
embeddings = model_emb.encode(text_list, show_progress_bar=True)

print("Applying PCA...")
pca = PCA(n_components=50, random_state=42)
embeddings_reduced = pca.fit_transform(embeddings)

print("Embeddings shape:", embeddings_reduced.shape)


 Combine Features

In [ ]:
numeric_features = [
    'Views','Likes','Shares','Comments',
    'like_rate','share_rate','comment_rate',
    'Platform_encoded','Content_Type_encoded','Region_encoded',
    'DayOfWeek','Month','Year','IsWeekend'
]

X_numeric = train[numeric_features].values
X = np.hstack([X_numeric, embeddings_reduced])
y = train['Viral'].values

print("Features combined")
print("X shape:", X.shape)
print("y shape:", y.shape)


 Train/Test Split

In [ ]:

categorical_cols = ['Platform', 'Content_Type', 'Region']
for col in categorical_cols:
    valid[col] = valid[col].astype(str).str.lower()

for col in categorical_cols:
    valid[col+'_encoded'] = le_dict[col].transform(valid[col])

valid['Post_Date'] = pd.to_datetime(valid['Post_Date'], errors='coerce')
median_date = train['Post_Date'].median()
valid['Post_Date'] = valid['Post_Date'].fillna(median_date)

valid['DayOfWeek'] = valid['Post_Date'].dt.dayofweek
valid['Month'] = valid['Post_Date'].dt.month
valid['Year'] = valid['Post_Date'].dt.year
valid['IsWeekend'] = valid['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

valid['like_rate'] = valid['Likes'] / valid['Views']
valid['share_rate'] = valid['Shares'] / valid['Views']
valid['comment_rate'] = valid['Comments'] / valid['Views']
valid[['like_rate','share_rate','comment_rate']] = valid[['like_rate','share_rate','comment_rate']].fillna(0)
valid[['like_rate','share_rate','comment_rate']] = scaler.transform(
    valid[['like_rate','share_rate','comment_rate']]
)

text_list_valid = valid["Pseudo_Caption"].fillna("").astype(str).tolist()
embeddings_valid = model_emb.encode(text_list_valid, show_progress_bar=True)
embeddings_valid_reduced = pca.transform(embeddings_valid)

numeric_features = [
    'Views','Likes','Shares','Comments',
    'like_rate','share_rate','comment_rate',
    'Platform_encoded','Content_Type_encoded','Region_encoded',
    'DayOfWeek','Month','Year','IsWeekend'
]

X_valid_numeric = valid[numeric_features].values
X_valid = np.hstack([X_valid_numeric, embeddings_valid_reduced])

print("Validation features ready")
print("X_valid shape:", X_valid.shape)
